In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from collections import defaultdict
from tqdm import tqdm  # Progress bar

In [ ]:
N_BINS = 10

low  = np.array([-1, -1, -1, -1, -12.567, -28.274])
high = np.array([ 1,  1,  1,  1,  12.567,  28.274])

def discretize(obs):
    obs = np.array(obs).flatten()
    ratios = (obs - low) / (high - low)
    idx = (ratios * N_BINS).astype(int)
    idx = np.clip(idx, 0, N_BINS - 1)
    return tuple(idx.tolist())  # .tolist() converts numpy ints to python ints\

In [ ]:
class AcrobatSarsaAgent:
    def __init__(
        self,
        env: gym.Env,
        learning_rate: float,
        initial_epsilon: float,
        epsilon_decay: float,
        final_epsilon: float,
        discount_factor: float = 0.99,
    ):
        """Initialize a Sarsa agent.

        Args:
            env: The training environment
            learning_rate: How quickly to update Q-values (0-1)
            initial_epsilon: Starting exploration rate (usually 1.0)
            epsilon_decay: How much to reduce epsilon each episode
            final_epsilon: Minimum exploration rate (usually 0.1)
            discount_factor: How much to value future rewards (0-1)
        """
        self.env = env

        # Q-table: maps (state, action) to expected reward
        # defaultdict automatically creates entries with zeros for new states
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        self.lr = learning_rate
        self.discount_factor = discount_factor  # How much we care about future rewards

        # Exploration parameters
        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon

        # Track learning progress
        self.training_error = []

    def get_action(self, obs: tuple[int, int, bool]) -> int:
        """Choose an action using epsilon-greedy strategy.

        Returns:
            action: 0 (stand) or 1 (hit)
        """
        # With probability epsilon: explore (random action)
        if np.random.random() < self.epsilon:
            return self.env.action_space.sample()

        # With probability (1-epsilon): exploit (best known action)
        else:
            return int(np.argmax(self.q_values[obs]))

    def update(
        self,
        obs: tuple[int, int, bool],
        action: int,
        reward: float,
        terminated: bool,
        next_obs: tuple[int, int, bool],
        next_action: int
    ):
        """Update Q-value based on experience.

        This is the heart of Sarsa: learn from (state, action, reward, next_state)
        """
        # What's the best we could do from the next state?
        # (Zero if episode terminated - no future rewards possible)
        future_q_value = (not terminated) * self.q_values[next_obs][next_action]

        # What should the Q-value be? (Bellman equation)
        target = reward + self.discount_factor * future_q_value

        # How wrong was our current estimate?
        temporal_difference = target - self.q_values[obs][action]

        # Update our estimate in the direction of the error
        # Learning rate controls how big steps we take
        self.q_values[obs][action] = (
            self.q_values[obs][action] + self.lr * temporal_difference
        )

        # Track learning progress (useful for debugging)
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        """Reduce exploration rate after each episode."""
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)

In [ ]:
class AcrobatQlearningAgent:
    def __init__(
        self,
        env: gym.Env,
        learning_rate: float,
        initial_epsilon: float,
        epsilon_decay: float,
        final_epsilon: float,
        discount_factor: float = 0.99,
    ):
        """Initialize a Q-Learning agent.

        Args:
            env: The training environment
            learning_rate: How quickly to update Q-values (0-1)
            initial_epsilon: Starting exploration rate (usually 1.0)
            epsilon_decay: How much to reduce epsilon each episode
            final_epsilon: Minimum exploration rate (usually 0.1)
            discount_factor: How much to value future rewards (0-1)
        """
        self.env = env

        # Q-table: maps (state, action) to expected reward
        # defaultdict automatically creates entries with zeros for new states
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        self.lr = learning_rate
        self.discount_factor = discount_factor  # How much we care about future rewards

        # Exploration parameters
        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon

        # Track learning progress
        self.training_error = []

    def get_action(self, obs: tuple[int, int, bool]) -> int:
        """Choose an action using epsilon-greedy strategy.

        Returns:
            action: 0 (stand) or 1 (hit)
        """
        # With probability epsilon: explore (random action)
        if np.random.random() < self.epsilon:
            return self.env.action_space.sample()

        # With probability (1-epsilon): exploit (best known action)
        else:
            return int(np.argmax(self.q_values[obs]))

    def update(
        self,
        obs: tuple[int, int, bool],
        action: int,
        reward: float,
        terminated: bool,
        next_obs: tuple[int, int, bool],
    ):
        """Update Q-value based on experience.

        This is the heart of Q-learning: learn from (state, action, reward, next_state)
        """
        # What's the best we could do from the next state?
        # (Zero if episode terminated - no future rewards possible)
        future_q_value = (not terminated) * np.max(self.q_values[next_obs])

        # What should the Q-value be? (Bellman equation)
        target = reward + self.discount_factor * future_q_value

        # How wrong was our current estimate?
        temporal_difference = target - self.q_values[obs][action]

        # Update our estimate in the direction of the error
        # Learning rate controls how big steps we take
        self.q_values[obs][action] = (
            self.q_values[obs][action] + self.lr * temporal_difference
        )

        # Track learning progress (useful for debugging)
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        """Reduce exploration rate after each episode."""
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)

### Write Python code for SARSA and Q-learning using ϵ−greedy exploration in the above environment (3 MARKS)

In [ ]:
env = gym.make('Acrobot-v1',render_mode="rgb_array")
env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=10000) 
agent = AcrobatSarsaAgent(
    env=env,
    learning_rate=0.1,
    initial_epsilon=0.1,
    #epsilon_decay=epsilon_decay,
    final_epsilon=0.1,
)

for episode in tqdm(range(10000), desc=f"lr={0.1}, eps={0.1}"):
    obs, info = env.reset()
    obs = discretize(obs)
    action = agent.get_action(obs)
    done = False
    while not done:
        # Take action and observe result
        next_obs, reward, terminated, truncated, info = env.step(action)
        next_obs = discretize(next_obs)
        next_action = agent.get_action(next_obs)

        # Learn from this experience
        agent.update(obs, action, reward, terminated, next_obs,next_action)

        # Move to next state
        done = terminated or truncated
        obs = next_obs
        action = next_action
    # epsilon decay
    #agent.epsilon = max(0.1, agent.epsilon - (eps / (n_episodes / 2)))

# store average reward of last 100 episodes as performance metric
avg_reward = np.mean(list(env.return_queue)[-100:])
print(f"avg_reward={avg_reward:.2f}")

In [ ]:
env = gym.make('Acrobot-v1',render_mode="rgb_array")
env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=10000) 
agent = AcrobatQlearningAgent(
    env=env,
    learning_rate=0.1,
    initial_epsilon=0.1,
    epsilon_decay=0.001,
    final_epsilon=0.1,
)

for episode in tqdm(range(10000), desc=f"lr={0.1}, eps={0.1}"):
    obs, info = env.reset()
    obs = discretize(obs)
    done = False
    while not done:
        # Agent chooses action (initially random, gradually more intelligent)
        action = agent.get_action(obs)
        
        # Take action and observe result
        next_obs, reward, terminated, truncated, info = env.step(action)
        next_obs = discretize(next_obs)

        # Learn from this experience
        agent.update(obs, action, reward, terminated, next_obs)

        # Move to next state
        done = terminated or truncated
        obs = next_obs
        
avg_reward = np.mean(list(env.return_queue)[-100:])
print(f"avg_reward={avg_reward:.2f}")

#### For both algorithms, tune the hyperparameters (stepsize and ϵ) and report the top three ones (resulting in highest returns). Is a constant ϵ sufficient for exploration?If not, implement an appropriate ϵ−decay schedule (→ 0) to get a good policy

In [ ]:
# SARSA Hyperparameter Tuning
learning_rates = [0.2, 0.22, 0.25, 0.28, 0.3]
epsilons = [0.005, 0.01, 0.02]
n_episodes = 10_000 
from itertools import product

results = {}

for lr, eps in product(learning_rates, epsilons):    
    env = gym.make('Acrobot-v1',render_mode="rgb_array")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes) 
    agent = AcrobatSarsaAgent(
        env=env,
        learning_rate=lr,
        initial_epsilon=eps,
        #epsilon_decay=epsilon_decay,
        final_epsilon=0.1,
    )
    
    for episode in tqdm(range(n_episodes), desc=f"lr={lr}, eps={eps}"):
        obs, info = env.reset()
        obs = discretize(obs)
        action = agent.get_action(obs)
        done = False
        while not done:
            # Take action and observe result
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_obs = discretize(next_obs)
            next_action = agent.get_action(next_obs)

            # Learn from this experience
            agent.update(obs, action, reward, terminated, next_obs,next_action)

            # Move to next state
            done = terminated or truncated
            obs = next_obs
            action = next_action
        # epsilon decay
        #agent.epsilon = max(0.1, agent.epsilon - (eps / (n_episodes / 2)))
    
    # store average reward of last 100 episodes as performance metric
    avg_reward = np.mean(list(env.return_queue)[-100:])
    results[(lr, eps)] = avg_reward
    print(f"lr={lr}, eps={eps} → avg_reward={avg_reward:.2f}")
# Sort by reward value in descending order
top3 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]

print("Top 3 Hyperparameter Combinations:")
print("-" * 40)
for rank, ((lr, eps), reward) in enumerate(top3, 1):
    print(f"Rank {rank}: learning_rate={lr}, epsilon={eps} → avg_reward={reward:.2f}")

In [ ]:
# take best hyperparameters
best_sarsa     = (0.2,0.01)
decay_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01]
n_episodes  = 10_000
results_decay = {}

for decay in decay_rates:
    env = gym.make("Acrobot-v1")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
    
    agent = AcrobatSarsaAgent(
        env=env,
        learning_rate=best_sarsa[0],
        initial_epsilon=best_sarsa[1],
        epsilon_decay=decay,
        final_epsilon=0.01,
    )
    
    for episode in tqdm(range(n_episodes), desc=f"decay={decay}"):
        obs, info = env.reset()
        obs = discretize(obs)
        action = agent.get_action(obs)
        done = False
        while not done:
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_obs = discretize(next_obs)
            next_action = agent.get_action(next_obs)
            agent.update(obs, action, reward, terminated, next_obs,next_action)
            done = terminated or truncated
            obs = next_obs
            action = next_action
        
        agent.decay_epsilon()        
    
    avg_reward = np.mean(list(env.return_queue)[-100:])
    results_decay[decay] = avg_reward
    print(f"decay={decay} → avg_reward={avg_reward:.2f}")  
best_decay = max(results_decay, key=results_decay.get)
print(f"\nBest decay: {best_decay} → reward: {results_decay[best_decay]:.2f}")

In [ ]:
# Q-Learning Hyperparameter Tuning
learning_rates = [0.08, 0.1, 0.12, 0.15, 0.18]
epsilons = [0.05, 0.08, 0.1, 0.12]
n_episodes = 10_000 
from itertools import product

results = {}

for lr, eps in product(learning_rates, epsilons):   
    env = gym.make('Acrobot-v1',render_mode="rgb_array")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes) 
    agent = AcrobatQlearningAgent(
        env=env,
        learning_rate=lr,
        initial_epsilon=eps,
        epsilon_decay=0.001,
        final_epsilon=0.1,
    )
    
    for episode in tqdm(range(n_episodes), desc=f"lr={lr}, eps={eps}"):
        obs, info = env.reset()
        obs = discretize(obs)
        done = False
        while not done:
            # Agent chooses action (initially random, gradually more intelligent)
            action = agent.get_action(obs)
            
            # Take action and observe result
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_obs = discretize(next_obs)

            # Learn from this experience
            agent.update(obs, action, reward, terminated, next_obs)

            # Move to next state
            done = terminated or truncated
            obs = next_obs
        # epsilon decay
        #agent.epsilon = max(0.1, agent.epsilon - (eps / (n_episodes / 2)))
    
    # store average reward of last 100 episodes as performance metric
    avg_reward = np.mean(list(env.return_queue)[-100:])
    results[(lr, eps)] = avg_reward
    print(f"lr={lr}, eps={eps} → avg_reward={avg_reward:.2f}")
# Sort by reward value in descending order
top3 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]

print("Top 3 Hyperparameter Combinations:")
print("-" * 40)
for rank, ((lr, eps), reward) in enumerate(top3, 1):
    print(f"Rank {rank}: learning_rate={lr}, epsilon={eps} → avg_reward={reward:.2f}")

In [ ]:
# Take best from each
best_qlearning = (0.1, 0.05)    # lr, initial_eps
decay_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01]
n_episodes  = 10_000
results_decay = {}

for decay in decay_rates:
    env = gym.make("Acrobot-v1")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
    
    agent = AcrobatQlearningAgent(
        env=env,
        learning_rate=best_qlearning[0],
        initial_epsilon=best_qlearning[1],
        epsilon_decay=decay,
        final_epsilon=0.01,
    )
    
    for episode in tqdm(range(n_episodes), desc=f"decay={decay}"):
        obs, info = env.reset()
        obs = discretize(obs)
        done = False
        while not done:
            action = agent.get_action(obs)
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_obs = discretize(next_obs)
            agent.update(obs, action, reward, terminated, next_obs)
            done = terminated or truncated
            obs = next_obs
        
        agent.decay_epsilon()        
    
    avg_reward = np.mean(list(env.return_queue)[-100:])
    results_decay[decay] = avg_reward
    print(f"decay={decay} → avg_reward={avg_reward:.2f}")  
best_decay = max(results_decay, key=results_decay.get)
print(f"\nBest decay: {best_decay} → reward: {results_decay[best_decay]:.2f}")

In [ ]:
# running on top 3
def get_moving_avgs(arr, window, convolution_mode):
    """Compute moving average to smooth noisy data."""
    return np.convolve(
        np.array(arr).flatten(),
        np.ones(window),
        mode=convolution_mode
    ) / window

top3_ql = [
    (0.1, 0.05),
    (0.08, 0.12),
    (0.08, 0.1),
]

top3_sarsa = [
    (0.2, 0.01),
    (0.2, 0.005),
    (0.3, 0.005),
]

n_episodes = 10_000
rolling_length = 500

ql_curves    = {}
sarsa_curves = {}

# --- Q-Learning ---
for lr, eps in top3_ql:
    env = gym.make("Acrobot-v1")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
    
    agent = AcrobatQlearningAgent(
        env=env,
        learning_rate=lr,
        initial_epsilon=eps,
        epsilon_decay=0,       # constant epsilon, no decay
        final_epsilon=eps,     # stays fixed
    )
    
    for episode in tqdm(range(n_episodes), desc=f"QL lr={lr} eps={eps}"):
        obs, info = env.reset()
        obs = discretize(obs)
        done = False
        while not done:
            action = agent.get_action(obs)
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_obs = discretize(next_obs)
            agent.update(obs, action, reward, terminated, next_obs)
            done = terminated or truncated
            obs = next_obs
    
    ql_curves[(lr, eps)] = list(env.return_queue)

# --- SARSA ---
for lr, eps in top3_sarsa:
    env = gym.make("Acrobot-v1")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
    
    agent = AcrobatSarsaAgent(
        env=env,
        learning_rate=lr,
        initial_epsilon=eps,
        epsilon_decay=0,       # constant epsilon
        final_epsilon=eps,
    )
    
    for episode in tqdm(range(n_episodes), desc=f"SARSA lr={lr} eps={eps}"):
        obs, info = env.reset()
        obs = discretize(obs)
        action = agent.get_action(obs)
        done = False
        while not done:
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_obs = discretize(next_obs)
            next_action = agent.get_action(next_obs)
            agent.update(obs, action, reward, terminated, next_obs, next_action)
            action = next_action
            done = terminated or truncated
            obs = next_obs
    
    sarsa_curves[(lr, eps)] = list(env.return_queue)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q-Learning plot
for rank, (lr, eps) in enumerate(top3_ql, 1):
    smoothed = get_moving_avgs(ql_curves[(lr, eps)], rolling_length, "valid")
    axes[0].plot(smoothed, label=f"Rank {rank}: lr={lr}, ε={eps}")
axes[0].set_title("Q-Learning — Top 3 Constant ε")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Average Reward")
axes[0].legend()

# SARSA plot
for rank, (lr, eps) in enumerate(top3_sarsa, 1):
    smoothed = get_moving_avgs(sarsa_curves[(lr, eps)], rolling_length, "valid")
    axes[1].plot(smoothed, label=f"Rank {rank}: lr={lr}, ε={eps}")
axes[1].set_title("SARSA — Top 3 Constant ε")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Average Reward")
axes[1].legend()

plt.suptitle("Return vs Episodes — Top 3 Hyperparameter Combinations (Constant ε)")
plt.tight_layout()
plt.show()

#### epsilon decay

In [ ]:
# Best params with decay
best_ql_decay    = (0.1, 0.05, 0.005)   # lr, initial_eps, decay
best_sarsa_decay = (0.2, 0.01, 0.001)   # lr, initial_eps, decay

n_episodes     = 10_000
rolling_length = 500

# --- Q-Learning with decay ---
env = gym.make("Acrobot-v1")
env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)

agent_ql = AcrobatQlearningAgent(
    env=env,
    learning_rate=best_ql_decay[0],
    initial_epsilon=best_ql_decay[1],
    epsilon_decay=best_ql_decay[2],
    final_epsilon=0.01,
)

for episode in tqdm(range(n_episodes), desc="QL with decay"):
    obs, info = env.reset()
    obs = discretize(obs)
    done = False
    while not done:
        action = agent_ql.get_action(obs)
        next_obs, reward, terminated, truncated, info = env.step(action)
        next_obs = discretize(next_obs)
        agent_ql.update(obs, action, reward, terminated, next_obs)
        done = terminated or truncated
        obs = next_obs
    agent_ql.decay_epsilon()

ql_decay_curve = list(env.return_queue)

# --- SARSA with decay ---
env = gym.make("Acrobot-v1")
env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)

agent_sarsa = AcrobatSarsaAgent(
    env=env,
    learning_rate=best_sarsa_decay[0],
    initial_epsilon=best_sarsa_decay[1],
    epsilon_decay=best_sarsa_decay[2],
    final_epsilon=0.01,
)

for episode in tqdm(range(n_episodes), desc="SARSA with decay"):
    obs, info = env.reset()
    obs = discretize(obs)
    action = agent_sarsa.get_action(obs)
    done = False
    while not done:
        next_obs, reward, terminated, truncated, info = env.step(action)
        next_obs = discretize(next_obs)
        next_action = agent_sarsa.get_action(next_obs)
        agent_sarsa.update(obs, action, reward, terminated, next_obs, next_action)
        action = next_action
        done = terminated or truncated
        obs = next_obs
    agent_sarsa.decay_epsilon()

sarsa_decay_curve = list(env.return_queue)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))   

# Q-Learning: best constant vs decay
ql_best_constant = get_moving_avgs(ql_curves[(0.1, 0.05)], rolling_length, "valid")
ql_best_decay    = get_moving_avgs(ql_decay_curve,          rolling_length, "valid")
axes[0].plot(ql_best_constant, label="Constant ε=0.05", color="blue")
axes[0].plot(ql_best_decay,    label="Decay ε→0.01",    color="red")
axes[0].set_title("Q-Learning — Constant vs Decay ε")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Average Reward")
axes[0].legend()

# SARSA: best constant vs decay
sarsa_best_constant = get_moving_avgs(sarsa_curves[(0.2, 0.01)], rolling_length, "valid")
sarsa_best_decay    = get_moving_avgs(sarsa_decay_curve,          rolling_length, "valid")
axes[1].plot(sarsa_best_constant, label="Constant ε=0.01", color="orange")
axes[1].plot(sarsa_best_decay,    label="Decay ε→0.01",    color="green")
axes[1].set_title("SARSA — Constant vs Decay ε")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Average Reward")
axes[1].legend()

plt.suptitle("Constant ε vs Decaying ε")
plt.tight_layout()
plt.show()

### Plot (with tuned hyperparameters) the mean performance along with confidence intervals over 10 random seeds/runs (sample plot shown below in Fig 1). A random seed corresponds to the initial random Q-values and the initial random start state. Explain the results along with comparing the two algorithms. 

In [ ]:
# part 2-b
def get_moving_avgs(arr, window, convolution_mode):
    """Compute moving average to smooth noisy data."""
    return np.convolve(
        np.array(arr).flatten(),
        np.ones(window),
        mode=convolution_mode
    ) / window

n_seeds = 10
n_episodes = 20_000
rolling_length = 500

# Best hyperparameters found
best_qlearning = (0.1, 0.05, 0.001)   # lr, initial_eps, decay
best_sarsa     = (0.2, 0.01, 0.005)   # lr, initial_eps, decay

def run_agent(agent_class, lr, initial_eps, decay, n_episodes, seed, is_sarsa=False):
    env = gym.make("Acrobot-v1")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
    
    np.random.seed(seed)  # set seed for reproducibility
    
    agent = agent_class(
        env=env,
        learning_rate=lr,
        initial_epsilon=initial_eps,
        epsilon_decay=decay,
        final_epsilon=0.01,
    )
    
    for episode in range(n_episodes):
        obs, info = env.reset(seed=seed+episode)  # different start state each episode
        obs = discretize(obs)
        done = False
        
        if is_sarsa:
            action = agent.get_action(obs)
        
        while not done:
            if is_sarsa:
                next_obs, reward, terminated, truncated, info = env.step(action)
                next_obs = discretize(next_obs)
                next_action = agent.get_action(next_obs)
                agent.update(obs, action, reward, terminated, next_obs, next_action)
                action = next_action
            else:
                action = agent.get_action(obs)
                next_obs, reward, terminated, truncated, info = env.step(action)
                next_obs = discretize(next_obs)
                agent.update(obs, action, reward, terminated, next_obs)
            
            done = terminated or truncated
            obs = next_obs
        
        agent.decay_epsilon()
    
    return list(env.return_queue)

# --- Collect rewards for all seeds ---
ql_all_rewards   = []
sarsa_all_rewards = []

for seed in tqdm(range(n_seeds), desc="Running seeds"):
    ql_rewards    = run_agent(AcrobatQlearningAgent, *best_qlearning, n_episodes, seed, is_sarsa=False)
    sarsa_rewards = run_agent(AcrobatSarsaAgent,     *best_sarsa,     n_episodes, seed, is_sarsa=True)
    ql_all_rewards.append(ql_rewards)
    sarsa_all_rewards.append(sarsa_rewards)

def smooth_all(all_rewards, window):
    return np.array([
        get_moving_avgs(run, window, "valid")
        for run in all_rewards
    ])

ql_smoothed    = smooth_all(ql_all_rewards,    rolling_length)
sarsa_smoothed = smooth_all(sarsa_all_rewards, rolling_length)

# --- Plot both together ---
fig, ax = plt.subplots(figsize=(10, 5))

for smoothed, label, color in zip(
    [ql_smoothed, sarsa_smoothed],
    ["Q-Learning", "SARSA"],
    ["blue", "orange"]
):
    mean = np.mean(smoothed, axis=0)
    std  = np.std(smoothed, axis=0)
    x    = range(len(mean))

    ax.plot(x, mean, color=color, label=f"{label} mean")
    ax.fill_between(x, mean - std, mean + std, alpha=0.2, color=color, label=f"{label} ±1 std")

ax.set_title(f"Q-Learning vs SARSA — Mean ± Std over {n_seeds} seeds")
ax.set_xlabel("Episode")
ax.set_ylabel("Average Reward")
ax.legend()
plt.tight_layout()
plt.show()

### Run the two algorithms starting with ϵ = 1, decaying ϵ till 0.1, and fixed thereafter.Compare (i) the online performance (performance while learning) and (ii) the performance of the policies after finishing learning (i.e., without any exploration). Explain the differences, if there exist any.

In [ ]:
n_episodes = 10_000

# Train Q-Learning
env_ql = gym.make("Acrobot-v1")
env_ql = gym.wrappers.RecordEpisodeStatistics(env_ql, buffer_length=n_episodes)

agent_ql = AcrobatQlearningAgent(
    env=env_ql,
    learning_rate=0.1,
    initial_epsilon=1.0,      # ← start at 1
    epsilon_decay=0.0001,
    final_epsilon=0.1,        # ← decay till 0.1, fixed after
)

for episode in tqdm(range(n_episodes), desc="Q-Learning"):
    obs, info = env_ql.reset()
    obs = discretize(obs)
    done = False
    while not done:
        action = agent_ql.get_action(obs)
        next_obs, reward, terminated, truncated, info = env_ql.step(action)
        next_obs = discretize(next_obs)
        agent_ql.update(obs, action, reward, terminated, next_obs)
        done = terminated or truncated
        obs = next_obs
    agent_ql.decay_epsilon()

# Train SARSA
env_sarsa = gym.make("Acrobot-v1")
env_sarsa = gym.wrappers.RecordEpisodeStatistics(env_sarsa, buffer_length=n_episodes)

agent_sarsa = AcrobatSarsaAgent(
    env=env_sarsa,
    learning_rate=0.2,
    initial_epsilon=1.0,      # ← start at 1
    epsilon_decay=0.0001,
    final_epsilon=0.1,        # ← decay till 0.1, fixed after
)

for episode in tqdm(range(n_episodes), desc="SARSA"):
    obs, info = env_sarsa.reset()
    obs = discretize(obs)
    action = agent_sarsa.get_action(obs)
    done = False
    while not done:
        next_obs, reward, terminated, truncated, info = env_sarsa.step(action)
        next_obs = discretize(next_obs)
        next_action = agent_sarsa.get_action(next_obs)
        agent_sarsa.update(obs, action, reward, terminated, next_obs, next_action)
        action = next_action
        done = terminated or truncated
        obs = next_obs
    agent_sarsa.decay_epsilon()

In [ ]:
rolling_length = 500
fig, ax = plt.subplots(figsize=(10, 5))

ql_avg = get_moving_avgs(env_ql.return_queue, rolling_length, "valid")
sarsa_avg = get_moving_avgs(env_sarsa.return_queue, rolling_length, "valid")

ax.plot(ql_avg, label="Q-Learning", color="blue")
ax.plot(sarsa_avg, label="SARSA", color="orange")
ax.set_title("Online Performance (during learning)")
ax.set_xlabel("Episode")
ax.set_ylabel("Average Reward")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluate Final Policy (epsilon=0) ---
def evaluate_policy(agent, n_eval=500):
    env = gym.make("Acrobot-v1")
    agent.epsilon = 0.0    # pure greedy, no exploration
    rewards = []
    for ep in range(n_eval):
        obs, info = env.reset()
        obs = discretize(obs)
        done = False
        total = 0
        while not done:
            action = agent.get_action(obs)
            next_obs, reward, terminated, truncated, info = env.step(action)
            obs = discretize(next_obs)
            total += reward
            done = terminated or truncated
        rewards.append(total)
    return np.mean(rewards), np.std(rewards)

ql_mean, ql_std = evaluate_policy(agent_ql)
sarsa_mean, sarsa_std = evaluate_policy(agent_sarsa)

# --- Plot Policy Performance ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(["Q-Learning", "SARSA"],
       [ql_mean, sarsa_mean],
       yerr=[ql_std, sarsa_std],
       color=["blue", "orange"], alpha=0.7, capsize=10)
ax.set_title("Policy Performance (after learning, ε=0)")
ax.set_ylabel("Average Reward")
plt.tight_layout()
plt.show()

print(f"Q-Learning  policy: {ql_mean:.2f} ± {ql_std:.2f}")
print(f"SARSA       policy: {sarsa_mean:.2f} ± {sarsa_std:.2f}")

## Intuitively, increasing the number of bins results in better state representation, enhancing granularity. This can result in learning a better policy. Is this always the case?Are there any downsides? Explain (try 5, 15, and 20 bins).

In [ ]:
# part -4

bin_sizes = [5, 15, 20]
n_episodes = 10_000

# best params
best_ql    = (0.1, 0.05, 0.001, 0.1)   # lr, initial_eps, decay, final_eps
best_sarsa = (0.2, 0.01, 0.005, 0.1)

ql_bin_results    = {}
sarsa_bin_results = {}

for n_bins in bin_sizes:
    # redefine discretize with new bin size
    def discretize(obs):
        obs = np.array(obs).flatten()
        ratios = (obs - low) / (high - low)
        idx = (ratios * n_bins).astype(int)
        idx = np.clip(idx, 0, n_bins - 1)
        return tuple(idx.tolist())

    # --- Q-Learning ---
    env = gym.make("Acrobot-v1")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
    agent_ql = AcrobatQlearningAgent(
        env=env,
        learning_rate=best_ql[0],
        initial_epsilon=best_ql[1],
        epsilon_decay=best_ql[2],
        final_epsilon=best_ql[3],
    )
    for episode in tqdm(range(n_episodes), desc=f"QL bins={n_bins}"):
        obs, info = env.reset()
        obs = discretize(obs)
        done = False
        while not done:
            action = agent_ql.get_action(obs)
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_obs = discretize(next_obs)
            agent_ql.update(obs, action, reward, terminated, next_obs)
            done = terminated or truncated
            obs = next_obs
        agent_ql.decay_epsilon()
    ql_bin_results[n_bins] = list(env.return_queue)

    # --- SARSA ---
    env = gym.make("Acrobot-v1")
    env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
    agent_sarsa = AcrobatSarsaAgent(
        env=env,
        learning_rate=best_sarsa[0],
        initial_epsilon=best_sarsa[1],
        epsilon_decay=best_sarsa[2],
        final_epsilon=best_sarsa[3],
    )
    for episode in tqdm(range(n_episodes), desc=f"SARSA bins={n_bins}"):
        obs, info = env.reset()
        obs = discretize(obs)
        action = agent_sarsa.get_action(obs)
        done = False
        while not done:
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_obs = discretize(next_obs)
            next_action = agent_sarsa.get_action(next_obs)
            agent_sarsa.update(obs, action, reward, terminated, next_obs, next_action)
            action = next_action
            done = terminated or truncated
            obs = next_obs
        agent_sarsa.decay_epsilon()
    sarsa_bin_results[n_bins] = list(env.return_queue)

In [ ]:
# --- Plot ---
rolling_length = 500
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, bin_results, title in zip(
    axes,
    [ql_bin_results, sarsa_bin_results],
    ["Q-Learning", "SARSA"]
):
    for n_bins, rewards in bin_results.items():
        smoothed = get_moving_avgs(rewards, rolling_length, "valid")
        ax.plot(smoothed, label=f"bins={n_bins}")
    ax.set_title(f"{title} — Effect of Bin Size")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Average Reward")
    ax.legend()

plt.tight_layout()
plt.show()

# --- Print Summary ---
print("Q-Learning:")
for n_bins, rewards in ql_bin_results.items():
    print(f"  bins={n_bins} → avg_reward={np.mean(rewards[-100:]):.2f}")

print("SARSA:")
for n_bins, rewards in sarsa_bin_results.items():
    print(f"  bins={n_bins} → avg_reward={np.mean(rewards[-100:]):.2f}")